[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione-superiori/blob/main/02_primo_scout_automatico.ipynb)

# Notebook 2 — Il primo scout automatico

Nel primo notebook abbiamo guardato i dati. Ora costruiamo il primo modello predittivo: una formula semplice che stima il valore di mercato a partire da una sola informazione, l'`overall`.

In [ ]:
from IPython.display import HTML, display

def concept(title, body, kind="idea"):
    colors = {
        "idea": ("#e8f3ff", "#0b5394"),
        "math": ("#fff3cd", "#7a4d00"),
        "warning": ("#fdecea", "#a61b1b"),
        "task": ("#eaf7ea", "#226622"),
        "story": ("#f3e8ff", "#5b2a86")
    }
    bg, border = colors.get(kind, colors["idea"])
    display(HTML(f'''
    <div style="background:{bg}; border-left:6px solid {border}; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
    <b style="color:{border}; font-size:18px">{title}</b><br>{body}
    </div>
    '''))

def reveal(title, body):
    display(HTML(f'''
    <details style="background:#f7f7f7; border:1px solid #ddd; padding:12px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
      <summary style="cursor:pointer; font-weight:bold">{title}</summary>
      <div style="margin-top:10px">{body}</div>
    </details>
    '''))

In [ ]:
concept("Missione", "Costruire una prima regressione lineare: una retta che usa l'overall per prevedere il valore di mercato.", "story")

In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = list(DATA_DIR.rglob("players_22.csv")) or list(DATA_DIR.rglob("*players*.csv"))
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Dataset caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

In [ ]:
concept("Regressione", "Un problema di regressione consiste nel prevedere un numero. Qui il numero e' il valore di mercato del calciatore. Il modello riceve in ingresso una o piu' caratteristiche e restituisce una previsione numerica.", "idea")

## Prepariamo input e target

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[["overall"]]
y = df["value_mln_eur"]

X.head(), y.head()

In [ ]:
concept("Input e target", "Nel linguaggio del machine learning, gli input sono le informazioni che forniamo al modello. Il target e' cio' che vogliamo prevedere. In questo caso l'input e' overall, il target e' value_mln_eur.", "math")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Regressione lineare semplice</b>

<p>Quando l'input &egrave; una sola variabile $x$, la regressione lineare cerca la <b>retta</b> di equazione:</p>

$$\hat{y} \;=\; w \cdot x + b$$

<p>dove:</p>
<ul>
<li>$\hat{y}$ (si legge "y-cappello") &egrave; la <b>previsione</b> del modello.</li>
<li>$w$ &egrave; la <b>pendenza</b>: di quanto cresce $\hat{y}$ se $x$ aumenta di $1$.</li>
<li>$b$ &egrave; l'<b>intercetta</b>: il valore di $\hat{y}$ quando $x = 0$.</li>
</ul>

<p>Per un dato giocatore con valore reale $y_i$, l'errore commesso dal modello &egrave; il <b>residuo</b>:</p>

$$r_i \;=\; y_i - \hat{y}_i$$

<p>Lo scopo dell'algoritmo &egrave; scegliere $w$ e $b$ in modo che i residui siano <i>complessivamente</i> piccoli.</p>
</div>


## Alleniamo il modello

In [ ]:
model = LinearRegression()
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_
print(f"Formula imparata: valore ≈ {w:.2f} * overall + ({b:.2f})")

In [ ]:
concept("Parametri imparati", "La retta ha due parametri: pendenza w e intercetta b. All'inizio non sappiamo quali valori usare. L'algoritmo li sceglie cercando la retta che riduce l'errore medio sui dati disponibili.", "math")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; La funzione di costo</b>

<p>"Errore complessivamente piccolo" deve diventare un numero da minimizzare. La regressione lineare usa l'<b>errore quadratico medio</b> (MSE):</p>

$$\mathrm{MSE}(w, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl(y_i - (w \cdot x_i + b)\bigr)^2$$

<p>Perch&eacute; il <b>quadrato</b>?</p>
<ul>
<li>Cos&igrave; gli errori positivi e negativi non si cancellano fra loro.</li>
<li>Gli errori grandi pesano molto di pi&ugrave; degli errori piccoli (penalit&agrave; pi&ugrave; severa).</li>
</ul>

<p>L'algoritmo cerca la coppia $(w, b)$ che rende $\mathrm{MSE}$ pi&ugrave; piccolo possibile. Per la regressione lineare esiste una <b>formula chiusa</b> (il <i>metodo dei minimi quadrati</i>) che restituisce subito la soluzione &mdash; niente tentativi.</p>
</div>


## Visualizziamo la retta dello scout automatico

In [ ]:
x_line = np.linspace(df["overall"].min(), df["overall"].max(), 100).reshape(-1, 1)
y_line = model.predict(x_line)

plt.figure(figsize=(8,5))
plt.scatter(df["overall"], df["value_mln_eur"], alpha=0.22, label="Giocatori")
plt.plot(x_line, y_line, linewidth=3, label="Modello lineare")
plt.xlabel("Overall rating")
plt.ylabel("Valore [milioni di euro]")
plt.title("Prima regressione: valore previsto da overall")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Dove sbaglia il modello? I residui


In [ ]:
# Per ogni giocatore: residuo = valore reale - previsione
residui = y - model.predict(X)

plt.figure(figsize=(8,4))
plt.scatter(df["overall"], residui, alpha=0.25)
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.xlabel("Overall rating")
plt.ylabel("Residuo [milioni di euro]")
plt.title("Residui del modello: dove e di quanto sbaglia")
plt.grid(True, alpha=0.3)
plt.show()


<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Come si leggono i residui</b>

<p>Il grafico dei residui mostra l'errore $r_i = y_i - \hat{y}_i$ in funzione dell'input.</p>

<ul>
<li>Se i residui sono <b>sparsi attorno a zero</b> senza struttura, il modello sta lavorando bene.</li>
<li>Se mostrano un <b>andamento</b> (curva, ventaglio, gruppi), allora c'&egrave; informazione che la retta non riesce a catturare: forse serve un modello pi&ugrave; ricco.</li>
<li>Residui <b>molto grandi</b> isolati segnalano giocatori "particolari" (outlier) che meriterebbero un'analisi a parte.</li>
</ul>
</div>


In [ ]:
reveal("Domanda", "Secondo voi la retta sottostima o sovrastima i campioni? E cosa succede ai giocatori fortissimi, con overall molto alto?")

## Misuriamo l'errore

In [ ]:
y_pred = model.predict(X)
mae = mean_absolute_error(y, y_pred)
rmse = mean_squared_error(y, y_pred) ** 0.5
r2 = r2_score(y, y_pred)

print(f"Errore medio assoluto: {mae:.2f} milioni di euro")
print(f"RMSE: {rmse:.2f} milioni di euro")
print(f"R^2: {r2:.3f}")

In [ ]:
concept("Errore medio assoluto", "Se l'errore medio assoluto vale, per esempio, 3 milioni, significa che in media le previsioni sbagliano di circa 3 milioni di euro. Questa misura e' intuitiva perche' ha la stessa unita' di misura del target.", "math")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Tre metriche per giudicare un modello</b>

<p><b>MAE</b> (errore medio assoluto): media dei moduli dei residui. Stessa unit&agrave; di misura del target.</p>

$$\mathrm{MAE} \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl|\, y_i - \hat{y}_i \,\bigr|$$

<p><b>RMSE</b> (radice dell'errore quadratico medio): pi&ugrave; severo del MAE perch&eacute; penalizza di pi&ugrave; gli errori grandi.</p>

$$\mathrm{RMSE} \;=\; \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

<p><b>$R^2$</b> (coefficiente di determinazione): la frazione della variabilit&agrave; del target spiegata dal modello.</p>

$$R^2 \;=\; 1 \;-\; \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

<ul>
<li>$R^2 = 1$: previsioni perfette.</li>
<li>$R^2 = 0$: il modello non fa meglio che indovinare sempre la media.</li>
<li>$R^2 < 0$: il modello fa <i>peggio</i> della media (succede davvero, sui dati nuovi).</li>
</ul>
</div>


## Facciamo scouting su alcuni giocatori

In [ ]:
sample = df.sample(10, random_state=7).copy()
sample["predicted_value_mln_eur"] = model.predict(sample[["overall"]])
sample["error_mln_eur"] = sample["predicted_value_mln_eur"] - sample["value_mln_eur"]
sample[["short_name", "age", "overall", "potential", "value_mln_eur", "predicted_value_mln_eur", "error_mln_eur"]].round(2)

In [ ]:
reveal("Cosa dovremmo aver capito", "Abbiamo costruito un primo modello molto semplice. Funziona meglio del tirare a caso, ma e' limitato: uno scout vero non guarda solo l'overall. Nel prossimo notebook aggiungeremo altri indizi.")